In [1]:
import os
import sys
import glob

# Tự động nạp PySpark từ SPARK_HOME nếu có
if "SPARK_HOME" in os.environ:
    spark_python = os.path.join(os.environ["SPARK_HOME"], "python")
    if spark_python not in sys.path:
        sys.path.insert(0, spark_python)
    py4j_zips = glob.glob(os.path.join(spark_python, "lib", "py4j*.zip"))
    for py4j_zip in py4j_zips:
        if py4j_zip not in sys.path:
            sys.path.insert(1, py4j_zip)

from pyspark.sql import SparkSession

def get_spark():
    # Nhận diện môi trường Docker hay Host
    is_docker = os.path.exists("/.dockerenv")
    default_polaris = "http://polaris:8181" if is_docker else "http://localhost:8181"
    default_minio = "http://minio:9000" if is_docker else "http://localhost:9000"

    polaris_uri = os.getenv("POLARIS_URI", default_polaris)
    minio_endpoint = os.getenv("MINIO_ENDPOINT", default_minio)
    aws_access_key = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
    aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")

    builder = SparkSession.builder \
        .appName("OlistIcebergSpark") \
        .master("local[2]") \
        .config("spark.driver.memory", "1g") \
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0,org.apache.iceberg:iceberg-aws-bundle:1.6.0") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.lakehouse.type", "rest") \
        .config("spark.sql.catalog.lakehouse.uri", polaris_uri) \
        .config("spark.sql.catalog.lakehouse.warehouse", "s3://lakehouse/") \
        .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
        .config("spark.sql.catalog.lakehouse.s3.endpoint", minio_endpoint) \
        .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
        .config("spark.sql.catalog.lakehouse.s3.access-key-id", aws_access_key) \
        .config("spark.sql.catalog.lakehouse.s3.secret-access-key", aws_secret_key) \
        .config("spark.sql.defaultCatalog", "lakehouse")

    return builder.getOrCreate()

def test_spark_creates_bronze_namespace():
    spark = get_spark()
    
    # 1. Tạo namespace bronze
    spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.bronze;")
    
    # 2. Liệt kê namespaces
    namespaces_df = spark.sql("SHOW NAMESPACES IN lakehouse;")
    namespaces = [row[0] for row in namespaces_df.collect()]
    print(f"\n[SPARK] Danh sách namespaces trên Polaris Catalog: {namespaces}")
    assert "bronze" in namespaces, "Namespace bronze chưa tồn tại!"
    
    spark.stop()

print(">>> Bắt đầu chạy test_spark_creates_bronze_namespace()...")
test_spark_creates_bronze_namespace()
print(">>> [PASS] Kiểm thử Spark kết nối Polaris REST Catalog và Iceberg thành công!")


>>> Bắt đầu chạy test_spark_creates_bronze_namespace()...

[SPARK] Danh sách namespaces trên Polaris Catalog: ['bronze']
>>> [PASS] Kiểm thử Spark kết nối Polaris REST Catalog và Iceberg thành công!
